In [1]:
"""
Combined GEM Data Processing and Vulnerability Index Calculation
=================================================================
This script processes Global Exposure Model (GEM) building exposure data,
assigns vulnerability classes, and calculates integrated vulnerability indices.
Creates two sets of indicators: cost-weighted and dwelling-weighted.
The code access GEM data and mapping functions through the original code by Robin Middelanis.

https://github.com/rmiddelanis/global-unbreakable-model/blob/main/src/unbreakable/scenario/data_processing/gather_gem_data.py

"""

import json
import os
import re
import tqdm
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive

# ============================================================================
# CONFIGURATION
# ============================================================================
drive.mount('/content/drive')

# Google Drive Paths
BASE_DATA_DIR = Path("/content/drive/MyDrive/Resilient_Housing_Global_Regression/data_raw/GEM")
OUTPUT_DIR = Path("/content/drive/MyDrive/Resilient_Housing_Global_Regression/data_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# GitHub Data Source
GEM_RAW_BASE = "https://raw.githubusercontent.com/gem/global_exposure_model/da2131a3309d160514e66db89818690239d51dbd"
GEM_API_URL = "https://api.github.com/repos/gem/global_exposure_model/git/trees/da2131a3309d160514e66db89818690239d51dbd?recursive=1"

HAZUS_COUNTRIES = ['VIR', 'PRI', 'CAN', 'USA']
HAZARDS_TO_INTEGRATE = ['FLOOD', 'STORM_SURGE', 'TSUNAMI']
INTEGRATED_WATER_HAZARD = 'WATER'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ============================================================================
# PART 1: GEM DATA PROCESSING FUNCTIONS
# ============================================================================

def load_mapping(gem_fields_path_, vuln_class_mapping_):
    gem_fields = json.load(open(gem_fields_path_, 'r'))
    field_value_to_type_map = {
        v: k.lower()
        for k in gem_fields.keys()
        for l in gem_fields[k].keys()
        for v in gem_fields[k][l]
    }

    mapping_df = pd.read_excel(vuln_class_mapping_, header=0)
    mapping_df.drop('comment', inplace=True, axis=1)
    mapping_df.rename(columns={'combined': 'default'}, inplace=True)
    mapping_df.set_index(['lat_load_mat', 'lat_load_sys', 'height'], inplace=True)

    # Expand combined categories (e.g., "A+B" becomes separate "A" and "B" columns)
    for c in mapping_df.columns:
        if '+' in c:
            for c_ in c.split('+'):
                if len(c_) > 1:
                    mapping_df[c_] = mapping_df[c]
            mapping_df = mapping_df.drop(c, axis=1)

    mapping_df = mapping_df.sort_index()
    return mapping_df, field_value_to_type_map

def identify_gem_attribute_type(attribute, field_value_to_type_map, verbose=True):
    if len(attribute) == 0 and verbose:
        print("Warning: Empty attribute.")

    types = np.unique([
        field_value_to_type_map.get(field.split(':')[0], 'unknown')
        for field in attribute.split('+')
    ])

    if len(types) == 1 and 'unknown' in types and verbose:
        print(f"Warning: Unknown type for attribute {attribute}.")
    elif len(types) == 2 and 'unknown' in types:
        types = types[types != 'unknown']
    elif len(types) > 1 and verbose:
        print(f"Warning: Multiple types {types} for attribute {attribute}.")

    return types

def decode_taxonomy(taxonomy, field_value_to_type_map, keep_unknown=False, verbose=False):
    res = pd.DataFrame(
        {col: [[]] for col in ['lat_load_mat', 'lat_load_sys', 'height', 'unknown']},
        index=[taxonomy]
    )
    res.index.name = 'taxonomy'

    attribute_types = {
        attribute: identify_gem_attribute_type(attribute, field_value_to_type_map, verbose)
        for attribute in taxonomy.split('/')
    }

    for attribute, attribute_type in attribute_types.items():
        res.loc[[taxonomy], attribute_type] = (
            res.loc[taxonomy, attribute_type] +
            pd.DataFrame(index=[taxonomy], columns=attribute_type, data=[[[attribute]] * len(attribute_type)])
        )

    for col in res.columns:
        if len(res.loc[taxonomy, col]) == 0:
            res.loc[taxonomy, col] = np.nan
        elif len(res.loc[taxonomy, col]) == 1:
            res.loc[taxonomy, col] = res.loc[taxonomy, col][0]
        elif len(res.loc[taxonomy, col]) > 1:
            if res.loc[taxonomy, col][0] in ['MATO', 'UNK'] and 'UNK' in res.loc[taxonomy, col][0]:
                res.loc[taxonomy, col] = res.loc[taxonomy, col][0]
            elif verbose:
                print(f"Warning: Multiple attributes have been mapped to the same type for taxonomy {taxonomy}.")

    if keep_unknown:
        return res
    else:
        return res.drop('unknown', axis=1)

def assign_vulnerability(material, resistance_system, height, mapping, verbose=False):
    if material in mapping.index:
        if len(mapping.loc[material]) == 1:
            return mapping.loc[[material]].transpose().squeeze().rename('vulnerability')
        else:
            if type(resistance_system) is str and len(resistance_system) > 0:
                resistance_system = resistance_system.split('+')[0]

            if (material, resistance_system) in mapping.index:
                if len(mapping.loc[(material, resistance_system)]) == 1:
                    return mapping.loc[material, resistance_system].transpose().squeeze().rename('vulnerability')
                else:
                    if type(height) is str and len(height) > 0:
                        height = height.split(':')[1].split('+')[0].split('-' if '-' in height else ',')
                        if len(height) > 1 and len(height[1]) == 0 or len(height) == 1:
                            height = [height[0], height[0]]
                        try:
                            height = [int(h) for h in height]
                            for h_idx in mapping.loc[(material, resistance_system)].index:
                                if h_idx != 'default':
                                    h_range = sorted([int(h) for h in h_idx.split(':')[1].split(',')])
                                    if h_range[0] <= height[0] <= h_range[1] or h_range[0] <= height[1] <= h_range[1]:
                                        return mapping.loc[(material, resistance_system, h_idx)].transpose().squeeze().rename('vulnerability')
                        except ValueError as e:
                            if verbose:
                                print(f"Warning: could not parse height value {height} to integer. Using default value.")

                return mapping.loc[(material, resistance_system, 'default')].transpose().squeeze().rename('vulnerability')

            return mapping.loc[(material, 'default')].transpose().squeeze().rename('vulnerability')
    else:
        raise ValueError(f"Could not assign vulnerability for unknown material {material}.")

# ============================================================================
# STEP 1: GATHER GEM DATA
# ============================================================================

def gather_gem_data_github(hazus_gem_mapping_path_, gem_fields_path_,
                           vuln_class_mapping_, verbose=True):
    """
    Gathers and processes GEM data from GitHub while reproducing aaa.docx logic.
    """
    # Initialize an empty DataFrame
    gem = pd.DataFrame()

    vars_to_keep = {
        'ID_0': 'iso3',
        'NAME_0': 'country',
        'OCCUPANCY': 'building_type',
        'MACRO_TAXO': 'macro_taxonomy',
        'TAXONOMY': 'taxonomy',
        'BUILDINGS': 'n_buildings',
        'DWELLINGS': 'n_dwellings',
        'TOTAL_AREA_SQM': 'total_area_sqm',
        'TOTAL_REPL_COST_USD': 'total_replacement_cost',
        'COST_CONTENTS_USD': 'contents_cost',
        'COST_STRUCTURAL_USD': 'structural_cost',
        'COST_NONSTRUCTURAL_USD': 'nonstructural_cost',
    }

    index_vars = ['ID_0', 'NAME_0', 'OCCUPANCY', 'MACRO_TAXO', 'TAXONOMY']

    # --- Step 1: Fetch URLs from GitHub API ---
    if verbose: print("Fetching file list from GEM GitHub...")
    r = requests.get(GEM_API_URL).json()
    csv_urls = [f"{GEM_RAW_BASE}/{node['path']}" for node in r.get('tree', [])
                if node['path'].endswith('Exposure_Summary_Taxonomy.csv')]

    # --- Step 2: Stream and Filter Data ---
    if verbose: print(f"Streaming {len(csv_urls)} countries...")
    for url in tqdm.tqdm(csv_urls, disable=not verbose):
        try:
            df = pd.read_csv(url)

            # *** FILTER: Keep ONLY residential ('Res') data *** [Reproduced from original]
            if 'OCCUPANCY' in df.columns:
                # Use str.contains to be robust against 'RES ' or 'Residential'
                df = df[df['OCCUPANCY'].astype(str).str.contains('Res', case=False, na=False)].copy()

            if df.empty:
                continue

            # Check for missing variables (Reproduced np.setdiff1d logic)
            vars_diff = np.setdiff1d(list(vars_to_keep.keys()), df.columns)

            # Grouping logic (Reproduced)
            valid_cols = list(set(vars_to_keep.keys()) - set(vars_diff))
            # Ensure index_vars are present in valid_cols
            actual_index = [v for v in index_vars if v in valid_cols]

            df = df[valid_cols].groupby(actual_index).sum(numeric_only=True).reset_index()
            gem = pd.concat([gem, df], ignore_index=True)
        except Exception as e:
            if verbose: print(f" Error processing {url}: {e}")
            continue

    # Check if data was collected
    if gem.empty:
        if verbose: print("\nWarning: No 'Res' data found.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    gem.rename(vars_to_keep, axis=1, inplace=True)

    # --- Step 3: Taxonomy Cleaning (Reproduced) ---
    replace_strings = {
        s: s.replace('+', '-') for s in
        ['MIX(MUR+W)', 'MIX(MR+W)', 'MIX(S+CR)', 'MIX(MUR+CR)', 'MIX(W+EU)',
         'MIX(MUR+STRUB+W)', 'MIX(MUR+STDRE+W)', 'MIX(S+CR+PC)']
    }
    for s, r in replace_strings.items():
        gem['taxonomy'] = gem['taxonomy'].str.replace(s, r, regex=False)

    # --- Step 4: HAZUS Mapping (Reproduced) ---
    hazus_gem_mapping = pd.read_csv(hazus_gem_mapping_path_, index_col=0).astype(str)
    hazus_gem_mapping.loc['MH', 'gem_str'] = 'INF/'
    hazus_gem_mapping.loc['W3', 'gem_str'] = 'W/'
    hazus_gem_mapping.loc['W4', 'gem_str'] = 'W/'

    hazus_mask = gem.iso3.isin(HAZUS_COUNTRIES)
    if not gem[hazus_mask].empty:
        # Strict logic for USA/CAN taxonomy parsing
        gem.loc[hazus_mask, 'taxonomy'] = (
            gem.loc[hazus_mask, 'taxonomy'].apply(
                lambda x: hazus_gem_mapping.loc[str(x).split('-')[1].split('/')[0], 'gem_str']
                if '-' in str(x) else x
            )
        )

    # --- Step 5: Decoding & Vulnerability Assignment ---
    vulnerability_mapping, field_value_to_type_map = load_mapping(
        gem_fields_path_=gem_fields_path_,
        vuln_class_mapping_=vuln_class_mapping_
    )

    unique_tax_strings = gem.taxonomy.unique()
    decoded_tax_strings = pd.concat([
        decode_taxonomy(t, field_value_to_type_map, keep_unknown=False, verbose=False)
        for t in tqdm.tqdm(unique_tax_strings, desc="Decoding taxonomies", disable=not verbose)
    ])

    res = pd.merge(gem, decoded_tax_strings, how='left', on='taxonomy')

    # Special cases (Reproduced)
    res.loc[(res.lat_load_mat.isna()) &
            (res.lat_load_sys.apply(lambda x: 'LN' in x if type(x) is str else False)), "lat_load_mat"] = 'UNK'
    res.loc[(res.lat_load_mat.isna()) &
            (res.taxonomy.apply(lambda x: str(x).startswith('UNK'))), "lat_load_mat"] = 'UNK'

    vulnerability = res.apply(
        lambda x: assign_vulnerability(x.lat_load_mat, x.lat_load_sys, x.height, vulnerability_mapping, verbose=False), axis=1
    )

    merged = pd.concat([res, vulnerability], axis=1)

    # --- Step 6: Share Calculations (Reproduced) ---
    def calculate_shares(df, weight_col, label):
        if verbose: print(f"    Calculating {label} vulnerability shares...")
        shares_list = []
        for h_class in vulnerability.columns:
            s = df.groupby(['iso3', 'country', h_class])[weight_col].sum()
            s = s / df.groupby('iso3')[weight_col].sum()
            s = s.unstack().fillna(0)
            s.columns = pd.MultiIndex.from_product([[h_class], s.columns])
            shares_list.append(s)
        return pd.concat(shares_list, axis=1)

    v_shares_cost = calculate_shares(merged, 'total_replacement_cost', "cost-weighted")
    v_shares_dwell = calculate_shares(merged, 'n_dwellings', "dwelling-weighted")

    if verbose:
        print(f"✅ Success: Processed {len(merged):,} records for {gem.iso3.nunique()} countries.")

    return merged, v_shares_cost, v_shares_dwell

# ============================================================================
# STEP 2 & 3: CALCULATE INDICES
# ============================================================================

def calculate_vulnerability_indices(v_class_shares_raw, suffix='', verbose=True):
    if v_class_shares_raw.empty: return pd.DataFrame()
    df = v_class_shares_raw.copy()
    df.columns = ['_'.join(col).strip().upper() for col in df.columns.values]

    # Aggregated Indices
    hazards = set([c.split('_')[0] for c in df.columns if '_' in c])
    for hz in hazards:
        robust_col, median_col, fragile_col = f'{hz}_ROBUST', f'{hz}_MEDIAN', f'{hz}_FRAGILE'
        if robust_col in df.columns: df[f'{hz}_ROBUST_ONLY{suffix}'] = df[robust_col]
        if robust_col in df.columns and median_col in df.columns:
            df[f'{hz}_ROBUSTNESS{suffix}'] = df[median_col] + df[robust_col]
        elif median_col in df.columns: df[f'{hz}_ROBUSTNESS{suffix}'] = df[median_col]
        if fragile_col in df.columns: df[f'{hz}_VULNERABILITY{suffix}'] = df[fragile_col]

    # Water Integration
    if f'FLOOD_ROBUSTNESS{suffix}' in df.columns:
        df[f'{INTEGRATED_WATER_HAZARD}_ROBUSTNESS{suffix}'] = df[f'FLOOD_ROBUSTNESS{suffix}']
        df[f'{INTEGRATED_WATER_HAZARD}_VULNERABILITY{suffix}'] = df[f'FLOOD_VULNERABILITY{suffix}']
        if f'FLOOD_ROBUST_ONLY{suffix}' in df.columns:
            df[f'{INTEGRATED_WATER_HAZARD}_ROBUST_ONLY{suffix}'] = df[f'FLOOD_ROBUST_ONLY{suffix}']

    # Cleanup
    cols_to_drop = [c for c in df.columns if any(h in c for h in HAZARDS_TO_INTEGRATE) and suffix in c]
    cols_to_drop += [c for c in df.columns if any(c.endswith(s) for s in ['_FRAGILE', '_MEDIAN', '_ROBUST']) and not c.endswith(suffix)]
    return df.drop(columns=list(set(cols_to_drop)), errors='ignore')

In [3]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == '__main__':
    # Step 1: Gather
    gem_data, shares_cost, shares_dwell = gather_gem_data_github(
        BASE_DATA_DIR / "hazus-gem_mapping.csv",
        BASE_DATA_DIR / "gem_taxonomy_fields.json",
        BASE_DATA_DIR / "gem-to-vulnerability_mapping_per_hazard.xlsx"
    )

    # Step 2: Cost-weighted indices
    v_indices_cost = calculate_vulnerability_indices(shares_cost, suffix='')

    # Step 3: Dwelling-weighted indices
    v_indices_dwell = calculate_vulnerability_indices(shares_dwell, suffix='_wd')

    # Step 4: Combine
    v_combined = pd.merge(v_indices_cost, v_indices_dwell, left_index=True, right_index=True, how='outer')

    # Step 5: Save
    v_combined.to_csv(OUTPUT_DIR / "country_vulnerability_class_shares.csv")
    print(f"\n✅ Success. Processed {len(v_combined)} countries.")

Fetching file list from GEM GitHub...
Streaming 215 countries...


Decoding taxonomies: 100%|██████████| 3712/3712 [00:37<00:00, 99.61it/s]


    Calculating cost-weighted vulnerability shares...
    Calculating dwelling-weighted vulnerability shares...
✅ Success: Processed 9,571 records for 215 countries.

✅ Success. Processed 215 countries.
